# 09 - Laundering patterns, and what an account score means

The transaction model is fixed in notebook 08. This notebook asks two things before the score reaches the LLM: which laundering patterns the model finds, and how an account's score should be read. Nothing here changes the model or the threshold. Test scores describe the model and select nothing.

An account can be in several attempts and pattern types, so the pattern rows overlap. The pattern file does not list every labelled transaction, so labelled transactions and positives outside it get their own row.

In [1]:
%%time
import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, brier_score_loss

from skeptical_second_reader.data import (
    ACCOUNTS_CSV,
    PATTERNS_TXT,
    account_entities,
    account_key,
    account_labels,
    load_transactions,
    pattern_members,
    read_accounts,
    read_patterns,
)
from skeptical_second_reader.features import ALL_FEATURES, WINDOW_END
from skeptical_second_reader.model import fit_final, precision_threshold
from skeptical_second_reader.paths import DATA_INTERIM
from skeptical_second_reader.split import (
    N_FOLDS,
    cached_folds,
    cv_predict,
    fold_roles,
    split_groups,
)
from skeptical_second_reader.transactions import (
    TX_FEATURES,
    account_positions,
    noisy_or,
    transaction_cv,
    transaction_features,
    transaction_labels,
    transaction_table,
)

df = load_transactions()
labels = account_labels(df)
owners = account_entities(labels.index, read_accounts(ACCOUNTS_CSV))
patterns = read_patterns(PATTERNS_TXT)
groups = split_groups(owners, pattern_members(patterns))

X = pd.read_parquet(DATA_INTERIM / "experiment_features.parquet")[ALL_FEATURES]
y = labels.loc[X.index]
fold_of = cached_folds(labels, groups, DATA_INTERIM).loc[X.index]
table = transaction_table(df)
features = transaction_features(table)
tx_labels = transaction_labels(df)
sender, receiver = account_positions(table, X.index)
is_self = sender == receiver
positive = y == 1
saved = pd.read_parquet(DATA_INTERIM / "transaction_scores.parquet")
explained = pd.read_parquet(DATA_INTERIM / "transaction_shap.parquet")
assert saved.index.equals(X.index) and explained.index.equals(X.index)
print(f"accounts {len(X):,}  positives {y.sum():,}  transactions {len(table):,}")

accounts 515,078  positives 6,347  transactions 5,077,237
CPU times: total: 50.5 s
Wall time: 48.1 s


## 1. The rounds again

The transaction model is run again, because its models and transaction scores were not stored. Its account scores must equal those of notebook 08. A transaction's out-of-fold score comes from the round in which its sender was tested, so it was never fitted on. For the score reading in section 6, each round also fits an isotonic curve from validation scores to validation labels.

In [2]:
%%time
result = transaction_cv(features, tx_labels, table, y, fold_of, precision_threshold)
largest_gap = (result.scores - saved["score"]).abs().max()
assert largest_gap < 1e-6 and (result.flags == saved["flag"]).all()
print("scores and flags equal notebook 08: True")
final = cv_predict(X, y, fold_of, fit_final, precision_threshold)

sender_fold = fold_of.to_numpy()[sender]
tx_score = np.full(len(table), np.nan, dtype="float32")
n_high = pd.Series(0, index=X.index, name="n_high")
calibrated = pd.Series(np.nan, index=X.index, name="calibrated")
for k, model in enumerate(result.models):
    scores_k = model.predict_proba(features)[:, 1]
    tx_score[sender_fold == k] = scores_k[sender_fold == k]
    roles = fold_roles(fold_of, k).to_numpy()
    val, test = roles == "val", roles == "test"
    high = scores_k >= result.thresholds[k]
    counted = np.zeros(len(X), dtype="int64")
    np.add.at(counted, sender[high], 1)
    np.add.at(counted, receiver[high & ~is_self], 1)
    n_high[test] = counted[test]
    values = noisy_or(scores_k, sender, receiver, len(X))
    curve = IsotonicRegression(out_of_bounds="clip").fit(values[val], y[val])
    calibrated[test] = curve.predict(values[test])
assert not np.isnan(tx_score).any() and not calibrated.isna().any()
flagged = result.flags == 1
old_flagged = final.flags == 1
print("flagged:", int(flagged.sum()), " account model:", int(old_flagged.sum()))

scores and flags equal notebook 08: True


flagged: 3572  account model: 2339
CPU times: total: 1h 19s
Wall time: 4min 25s


## 2. Accounts by pattern type

Every account of a type with at least one transaction in the window, whether or not the transaction of the attempt itself falls in the window. All of them are positive. `not in the pattern file` is every other positive account. Recall per round shows how far a type depends on the split, with few accounts in some types.

In [3]:
members_all = pattern_members(patterns).merge(
    patterns.drop_duplicates("attempt")[["attempt", "pattern"]], on="attempt"
)
in_file = X.index.isin(members_all["account"])
print("share of pattern-file accounts that are positive:", y[in_file].mean())
print("accounts of the pattern file seen in the window:", int(in_file.sum()))

kinds = sorted(members_all["pattern"].unique())
types = {
    kind: X.index.isin(members_all.loc[members_all["pattern"] == kind, "account"])
    for kind in kinds
}
types["not in the pattern file"] = positive.to_numpy() & ~in_file
types["all positives"] = positive.to_numpy()
rows = []
for mask in types.values():
    per_round = [flagged[mask & (fold_of == k)].mean() for k in range(N_FOLDS)]
    part = result.scores[mask]
    rows.append(
        {
            "accounts": int(mask.sum()),
            "recall, account model": old_flagged[mask].mean(),
            "recall": flagged[mask].mean(),
            "round min": np.min(per_round),
            "round max": np.max(per_round),
            "score at least 0.9": (part >= 0.9).mean(),
            "flagged below 0.9": (flagged[mask] & (part < 0.9)).mean(),
            "median score": part.median(),
        }
    )
print(pd.DataFrame(rows, index=list(types)).round(3).to_string())

share of pattern-file accounts that are positive: 1.0
accounts of the pattern file seen in the window: 3160


                         accounts  recall, account model  recall  round min  round max  score at least 0.9  flagged below 0.9  median score
BIPARTITE                     491                  0.403   0.764      0.519      0.882               0.627              0.136         0.997
CYCLE                         270                  0.715   0.852      0.778      0.937               0.737              0.115         1.000
FAN-IN                        336                  0.580   0.667      0.486      0.796               0.577              0.089         0.997
FAN-OUT                       356                  0.385   0.823      0.635      0.971               0.761              0.062         1.000
GATHER-SCATTER                683                  0.422   0.612      0.461      0.792               0.552              0.060         0.995
RANDOM                        211                  0.569   0.777      0.652      0.976               0.664              0.114         1.000
SCATTER-GATHER      

Does the role in the attempt matter? Each account of the file is taken by its role in the labelled transactions of the window: it only sends, only receives, or does both. Accounts with no labelled transaction in the window form their own row. An account of several types counts in each.

In [4]:
bad = tx_labels.to_numpy() == 1
sends_bad = np.zeros(len(X), dtype=bool)
receives_bad = np.zeros(len(X), dtype=bool)
sends_bad[sender[bad & ~is_self]] = True
receives_bad[receiver[bad & ~is_self]] = True
self_bad = np.zeros(len(X), dtype=bool)
self_bad[sender[bad & is_self]] = True
roles_of = np.select(
    [sends_bad & receives_bad, sends_bad, receives_bad, self_bad],
    ["sends and receives", "only sends", "only receives", "only to itself"],
    default="no label in the window",
)
role = pd.Series(roles_of, index=X.index)
rows = []
for name, mask in types.items():
    for role_name, part in role[mask].groupby(role[mask]):
        rows.append(
            {
                "type": name,
                "role": role_name,
                "accounts": len(part),
                "recall": flagged[part.index].mean(),
            }
        )
by_role = pd.DataFrame(rows)
print(
    by_role.pivot(index="type", columns="role", values="accounts")
    .reindex(list(types))
    .fillna(0)
    .astype(int)
    .to_string()
)
print()
print(
    by_role.pivot(index="type", columns="role", values="recall")
    .reindex(list(types))
    .round(3)
    .to_string()
)

role                     no label in the window  only receives  only sends  sends and receives
type                                                                                          
BIPARTITE                                    33            204         212                  42
CYCLE                                        23             12          11                 224
FAN-IN                                       59             29         219                  29
FAN-OUT                                      57            225          33                  41
GATHER-SCATTER                              220            160         244                  59
RANDOM                                       34             28          24                 125
SCATTER-GATHER                               31             75          36                 226
STACK                                        29            206         207                 220
not in the pattern file                       0   

## 3. Attempts by pattern type

A reviewer may need one flagged account to find a whole attempt. Only attempts with a transaction in the window count, and only accounts from those window transactions. The split keeps an attempt in one fold, so one round scores all of it, unless an account also belongs to a group elsewhere; the groups of `split_groups` join those.

In [5]:
in_window = patterns[patterns["timestamp"] < WINDOW_END]
window_members = pattern_members(in_window).merge(
    patterns.drop_duplicates("attempt")[["attempt", "pattern"]], on="attempt"
)
window_members = window_members[window_members["account"].isin(X.index)]
window_members["flag"] = flagged.reindex(window_members["account"]).to_numpy()
window_members["old_flag"] = old_flagged.reindex(window_members["account"]).to_numpy()
window_members["fold"] = fold_of.reindex(window_members["account"]).to_numpy()
print(
    "attempts in the file:",
    patterns["attempt"].nunique(),
    " with a window transaction:",
    window_members["attempt"].nunique(),
)
print(
    "attempts whose accounts share one fold:",
    (window_members.groupby("attempt")["fold"].nunique() == 1).mean(),
)
per_attempt = window_members.groupby(["pattern", "attempt"]).agg(
    accounts=("flag", "size"),
    flagged=("flag", "sum"),
    old_flagged=("old_flag", "sum"),
)
per_attempt["share"] = per_attempt["flagged"] / per_attempt["accounts"]


def attempt_summary(frame):
    return frame.agg(
        attempts=("share", "size"),
        median_accounts=("accounts", "median"),
        any_flagged=("flagged", lambda s: (s > 0).mean()),
        any_flagged_account_model=("old_flagged", lambda s: (s > 0).mean()),
        median_share=("share", "median"),
        all_flagged=("share", lambda s: (s == 1).mean()),
    )


flat = per_attempt.reset_index()
summary = pd.concat(
    [
        attempt_summary(flat.groupby("pattern")),
        attempt_summary(flat.assign(pattern="all attempts").groupby("pattern")),
    ]
)
print(summary.round(3).to_string())

attempts in the file: 370  with a window transaction: 363
attempts whose accounts share one fold: 1.0
                attempts  median_accounts  any_flagged  any_flagged_account_model  median_share  all_flagged
pattern                                                                                                     
BIPARTITE             45              8.0        0.889                      0.733         0.857        0.467
CYCLE                 54              4.0        0.944                      0.833         1.000        0.741
FAN-IN                40              6.0        0.975                      0.900         0.888        0.450
FAN-OUT               48              5.5        0.896                      0.854         1.000        0.854
GATHER-SCATTER        51              5.0        0.882                      0.824         1.000        0.569
RANDOM                41              4.0        0.878                      0.780         1.000        0.756
SCATTER-GATHER        44  

## 4. Labelled transactions by pattern type

A labelled transaction of the window is matched to the file by time, sender and receiver. What do the labelled transactions of each type look like, and what score do they get? The share above 0.5 is out of fold.

In [6]:
tx_keys = pd.DataFrame(
    {
        "timestamp": table["timestamp"],
        "sender": table["sender"],
        "receiver": table["receiver"],
        "row": np.arange(len(table)),
    }
)[bad]
file_keys = in_window.assign(
    sender=account_key(in_window["from_bank"], in_window["from_account"]),
    receiver=account_key(in_window["to_bank"], in_window["to_account"]),
)[["timestamp", "sender", "receiver", "pattern"]].drop_duplicates(
    ["timestamp", "sender", "receiver"]
)
matched = tx_keys.merge(file_keys, on=["timestamp", "sender", "receiver"], how="left")
assert len(matched) == bad.sum()
print(
    "labelled window transactions:",
    len(matched),
    " found in the file:",
    int(matched["pattern"].notna().sum()),
)
print(
    "window rows of the file:",
    len(in_window),
    " of those matched to a labelled transaction:",
    int(file_keys.merge(tx_keys, on=["timestamp", "sender", "receiver"]).shape[0]),
)
matched["pattern"] = matched["pattern"].fillna("not in the pattern file")
rows_of = matched["row"].to_numpy()
matched["ach"] = table["payment_format"].to_numpy()[rows_of] == "ACH"
matched["one_off"] = features["pair_count"].to_numpy()[rows_of] == 1
matched["after_conversion"] = features["after_conversion"].to_numpy()[rows_of] == 1
matched["is_self"] = is_self[rows_of]
matched["score"] = tx_score[rows_of]
matched["above_half"] = matched["score"] >= 0.5
order = [*kinds, "not in the pattern file"]
print(
    matched.groupby("pattern")
    .agg(
        transactions=("score", "size"),
        ach=("ach", "mean"),
        one_off=("one_off", "mean"),
        after_conversion=("after_conversion", "mean"),
        self_tx=("is_self", "mean"),
        median_score=("score", "median"),
        above_half=("above_half", "mean"),
    )
    .reindex(order)
    .round(3)
    .to_string()
)

labelled window transactions: 4522  found in the file: 2554
window rows of the file: 2554  of those matched to a labelled transaction: 2554


                         transactions    ach  one_off  after_conversion  self_tx  median_score  above_half
pattern                                                                                                   
BIPARTITE                         241  1.000    0.925             0.606    0.000         0.995       0.664
CYCLE                             243  1.000    0.695             0.588    0.000         0.980       0.700
FAN-IN                            252  1.000    0.909             0.603    0.000         0.998       0.651
FAN-OUT                           276  1.000    0.920             0.717    0.000         1.000       0.906
GATHER-SCATTER                    462  1.000    0.887             0.662    0.017         1.000       0.768
RANDOM                            150  1.000    0.673             0.593    0.000         0.985       0.727
SCATTER-GATHER                    497  1.000    0.837             0.630    0.000         0.999       0.815
STACK                             433

## 5. What the score bands hold

All accounts by band. `below the threshold` is the unflagged accounts, whatever their score, and the flagged bands start at the threshold of the account's round. The range shows the positive rate per round. `strongest` is the account's highest-scored transaction. `high transactions` counts its transactions at or above the threshold.

In [7]:
in_conversions = np.zeros(len(X), dtype="int64")
converted = is_self & (features["changes_currency"].to_numpy() == 1)
np.add.at(in_conversions, sender[converted], 1)
is_ach = table["payment_format"].to_numpy() == "ACH"
after_conversion = features["after_conversion"].to_numpy() == 1
sends_after = np.zeros(len(X), dtype=bool)
sends_after[sender[after_conversion & is_ach & ~is_self]] = True
rule = pd.Series(sends_after & (in_conversions == 1), index=X.index)

strongest = result.strongest.to_numpy()
strongest_one_off = pd.Series(
    features["pair_count"].to_numpy()[strongest] == 1, index=X.index
)
strongest_after = pd.Series(
    features["after_conversion"].to_numpy()[strongest] == 1, index=X.index
)
strongest_ach = pd.Series(is_ach[strongest], index=X.index)
activity = np.zeros(len(X), dtype="int64")
np.add.at(activity, sender, 1)
np.add.at(activity, receiver[~is_self], 1)
activity = pd.Series(activity, index=X.index)

BAND_NAMES = [
    "below the threshold",
    "flagged, below 0.5",
    "0.5 to 0.9",
    "0.9 to 0.99",
    "0.99 and above",
]
score_of = result.scores
band = pd.Series(BAND_NAMES[0], index=X.index)
band[flagged & (score_of < 0.5)] = BAND_NAMES[1]
band[flagged & (score_of >= 0.5) & (score_of < 0.9)] = BAND_NAMES[2]
band[flagged & (score_of >= 0.9) & (score_of < 0.99)] = BAND_NAMES[3]
band[flagged & (score_of >= 0.99)] = BAND_NAMES[4]
by_round = y.groupby([band, fold_of]).mean().unstack()
rows = []
for name in BAND_NAMES:
    mask = band == name
    rows.append(
        {
            "accounts": int(mask.sum()),
            "positives": int(y[mask].sum()),
            "positive rate": y[mask].mean(),
            "round min": by_round.loc[name].min(),
            "round max": by_round.loc[name].max(),
            "inside the rule": rule[mask].mean(),
            "strongest one-off": strongest_one_off[mask].mean(),
            "strongest after conversion": strongest_after[mask].mean(),
            "strongest ACH": strongest_ach[mask].mean(),
            "median high transactions": n_high[mask].median(),
            "median transactions": activity[mask].median(),
        }
    )
print(pd.DataFrame(rows, index=BAND_NAMES).round(4).T.to_string())
print()
share = pd.crosstab(role[positive], band[positive], normalize="index")[BAND_NAMES]
print("positives by role in the labelled transactions and band")
print(share.round(3).to_string())

                            below the threshold  flagged, below 0.5  0.5 to 0.9  0.9 to 0.99  0.99 and above
accounts                            511506.0000            873.0000    510.0000     231.0000       1958.0000
positives                             3889.0000            134.0000    192.0000     203.0000       1929.0000
positive rate                            0.0076              0.1535      0.3765       0.8788          0.9852
round min                                0.0072              0.1061      0.2437       0.8462          0.9763
round max                                0.0079              0.1885      0.4928       0.9259          0.9891
inside the rule                          0.0001              0.0103      0.0686       0.2554          0.5230
strongest one-off                        0.4433              0.8603      0.8549       0.8788          0.9648
strongest after conversion               0.0039              0.0298      0.1471       0.4848          0.9683
strongest ACH      

Within the flagged accounts below 0.9, what tells a true positive from a false alarm? The same question for the explanation: does the largest SHAP contribution of the strongest transaction change with the band? If it does not, the reader gets the same reason at very different levels of certainty.

In [8]:
below = flagged & (score_of < 0.9)
outcome_masks = {
    "true positives below 0.9": below & positive,
    "false alarms below 0.9": below & ~positive,
    "true positives at 0.9+": flagged & (score_of >= 0.9) & positive,
    "false alarms at 0.9+": flagged & (score_of >= 0.9) & ~positive,
}
rows = {
    name: {
        "accounts": int(mask.sum()),
        "strongest one-off": strongest_one_off[mask].mean(),
        "strongest after conversion": strongest_after[mask].mean(),
        "median high transactions": n_high[mask].median(),
        "only one high transaction": (n_high[mask] == 1).mean(),
        "no high transaction": (n_high[mask] == 0).mean(),
        "median transactions": activity[mask].median(),
        "median score": score_of[mask].median(),
    }
    for name, mask in outcome_masks.items()
}
print(pd.DataFrame(rows).round(3).to_string())

contributions = explained[TX_FEATURES]
largest = contributions[flagged].abs().idxmax(axis=1)
by_band = pd.crosstab(band[flagged], largest, normalize="index").reindex(BAND_NAMES[1:])
print()
print("largest contribution by band, flagged accounts")
print(by_band.loc[:, by_band.max() >= 0.02].round(3).to_string())
order = np.argsort(-contributions[flagged].abs().to_numpy(), axis=1)[:, :3]
names = np.array(TX_FEATURES)[order]
combos = pd.Series(
    [", ".join(sorted(row)) for row in names], index=contributions[flagged].index
)
print()
print("distinct top-3 feature sets among flagged accounts:", combos.nunique())
print(combos.value_counts(normalize=True).head(5).round(3).to_string())
margin = contributions[flagged].sum(axis=1)
print()
print("sum of SHAP values of the strongest transaction by band (median)")
print(
    margin.groupby(band[flagged]).median().reindex(BAND_NAMES[1:]).round(3).to_string()
)

                            true positives below 0.9  false alarms below 0.9  true positives at 0.9+  false alarms at 0.9+
accounts                                     326.000                1057.000                2132.000                57.000
strongest one-off                              0.893                   0.848                   0.957                 0.912
strongest after conversion                     0.129                   0.056                   0.921                 0.789
median high transactions                       1.000                   1.000                   1.000                 1.000
only one high transaction                      0.822                   0.658                   0.675                 0.789
no high transaction                            0.117                   0.327                   0.007                 0.018
median transactions                           28.000                  23.000                  24.000                13.000
median score    

## 6. Reading the score

Three ways to show a score to the reader are compared. The raw noisy-or. The same score through an isotonic curve fitted on the validation accounts of the round, so it can be read as a rate. And the bands of section 5, whose rates come from these same test accounts, so they describe but do not validate. The curve is judged on the test accounts.

In [9]:
bins = [0, 0.01, 0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 0.9, 1.0]
for name, values in {"raw": result.scores, "calibrated": calibrated}.items():
    binned = pd.cut(values, bins, include_lowest=True)
    table_of = pd.DataFrame(
        {
            "accounts": values.groupby(binned, observed=False).size(),
            "mean_score": values.groupby(binned, observed=False).mean(),
            "positive_rate": y.groupby(binned, observed=False).mean(),
        }
    )
    print(name)
    print(table_of.round(4).to_string())
    print()
measures = []
for k in range(N_FOLDS):
    test = fold_of == k
    measures.append(
        {
            "brier raw": brier_score_loss(y[test], result.scores[test]),
            "brier calibrated": brier_score_loss(y[test], calibrated[test]),
            "brier base rate": y[test].mean() * (1 - y[test].mean()),
            "auc_pr raw": average_precision_score(y[test], result.scores[test]),
            "auc_pr calibrated": average_precision_score(y[test], calibrated[test]),
            "distinct calibrated values": calibrated[test].nunique(),
            "mean calibrated at flagged": calibrated[test & flagged].mean(),
            "precision at flagged": y[test & flagged].mean(),
        }
    )
print(pd.DataFrame(measures).agg(["mean", "std"]).T.round(5).to_string())

flag_calibrated = pd.cut(
    calibrated[flagged], [0, 0.25, 0.5, 0.75, 0.9, 1.0], include_lowest=True
)
print()
print("flagged accounts by calibrated score")
print(
    y[flagged]
    .groupby(flag_calibrated, observed=False)
    .agg(["size", "sum", "mean"])
    .round(3)
    .to_string()
)

raw
                accounts  mean_score  positive_rate
score                                              
(-0.001, 0.01]    431808      0.0011         0.0019
(0.01, 0.05]       60013      0.0229         0.0276
(0.05, 0.1]        12848      0.0697         0.0627
(0.1, 0.2]          5610      0.1353         0.0834
(0.2, 0.4]          1826      0.2738         0.1046
(0.4, 0.6]           443      0.4845         0.2348
(0.6, 0.8]           238      0.6895         0.3739
(0.8, 0.9]           103      0.8497         0.5922
(0.9, 1.0]          2189      0.9956         0.9740

calibrated
                accounts  mean_score  positive_rate
calibrated                                         
(-0.001, 0.01]    422014      0.0016         0.0017
(0.01, 0.05]       67457      0.0243         0.0243
(0.05, 0.1]        20749      0.0668         0.0669
(0.1, 0.2]          1923      0.1183         0.1087
(0.2, 0.4]           501      0.2630         0.2375
(0.4, 0.6]           172      0.5241         0.4

                                mean      std
brier raw                    0.00804  0.00031
brier calibrated             0.00778  0.00031
brier base rate              0.01217  0.00001
auc_pr raw                   0.48058  0.02595
auc_pr calibrated            0.46408  0.02290
distinct calibrated values  94.60000  7.76531
mean calibrated at flagged   0.69457  0.01957
precision at flagged         0.68769  0.01413

flagged accounts by calibrated score
                size   sum   mean
calibrated                       
(-0.001, 0.25]   782   113  0.145
(0.25, 0.5]      408   100  0.245
(0.5, 0.75]      276   178  0.645
(0.75, 0.9]      115   105  0.913
(0.9, 1.0]      1991  1962  0.985


## 7. Two populations of positives

Section 2 shows a sharp split: accounts of the pattern file are found, other positives are not. Here each population is scored against all negatives, so the two AUC-PR values have different base rates and are not directly comparable; the base rate is shown with each. Accounts of the file with no labelled transaction in the window get their own row, since their laundering lies after the window. The last table asks where flagged accounts with no high transaction stand: they reach the threshold only because many small scores add up.

In [10]:
negative = y == 0
file_mask = pd.Series(in_file, index=X.index)
has_label = role != "no label in the window"
populations = {
    "pattern file": positive & file_mask,
    "pattern file, label in the window": positive & file_mask & has_label,
    "pattern file, no label in the window": positive & file_mask & ~has_label,
    "not in the pattern file": positive & ~file_mask,
}


def population_row(mask):
    scope = mask | negative
    tx_values, old_values = [], []
    for k in range(N_FOLDS):
        part = scope & (fold_of == k)
        tx_values.append(average_precision_score(mask[part], result.scores[part]))
        old_values.append(average_precision_score(mask[part], final.scores[part]))
    return {
        "accounts": int(mask.sum()),
        "share of positives": mask.sum() / positive.sum(),
        "base rate": mask.sum() / scope.sum(),
        "recall, account model": old_flagged[mask].mean(),
        "recall": flagged[mask].mean(),
        "share of true positives": (flagged & mask).sum() / (flagged & positive).sum(),
        "auc_pr, account model": np.mean(old_values),
        "auc_pr": np.mean(tx_values),
        "auc_pr sd": np.std(tx_values, ddof=1),
    }


print(
    pd.DataFrame({n: population_row(m) for n, m in populations.items()})
    .round(4)
    .to_string()
)

print()
print("flagged accounts by band: share of each kind of account")
kind_of = pd.Series("negative", index=X.index)
kind_of[positive & file_mask] = "pattern file"
kind_of[positive & ~file_mask] = "not in the pattern file"
print(pd.crosstab(band[flagged], kind_of[flagged]).reindex(BAND_NAMES[1:]).to_string())

print()
print("flagged accounts by number of high transactions")
high_bin = pd.cut(
    n_high[flagged], [-1, 0, 1, 3, np.inf], labels=["0", "1", "2-3", "4+"]
)
print(
    y[flagged]
    .groupby(high_bin, observed=True)
    .agg(["size", "sum", "mean"])
    .round(3)
    .to_string()
)

print()
print("labelled transactions by population")
ach_labelled = matched.assign(in_file=matched["pattern"] != "not in the pattern file")
print(
    ach_labelled.groupby("in_file")
    .agg(
        transactions=("score", "size"),
        ach=("ach", "mean"),
        one_off=("one_off", "mean"),
        after_conversion=("after_conversion", "mean"),
        median_score=("score", "median"),
        above_half=("above_half", "mean"),
    )
    .round(3)
    .to_string()
)

                         pattern file  pattern file, label in the window  pattern file, no label in the window  not in the pattern file
accounts                    3160.0000                          2675.0000                              485.0000                3187.0000
share of positives             0.4979                             0.4215                                0.0764                   0.5021
base rate                      0.0062                             0.0052                                0.0010                   0.0062
recall, account model          0.5000                             0.5903                                0.0021                   0.0223
recall                         0.7551                             0.8916                                0.0021                   0.0226
share of true positives        0.9707                             0.9703                                0.0004                   0.0293
auc_pr, account model          0.5534           